**Header**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE    = '/content/drive/MyDrive/churn_project/'
DATA    = BASE + 'data/raw/'
PROC    = BASE + 'data/processed/'
RESULTS = BASE + 'results/'

print("✓ Ready")

Mounted at /content/drive
✓ Ready


**Reload all 3 datasets:**

In [2]:
import zipfile

# ── Insurance: train.csv only ─────────────────────────────
# Paper used 33,908 rows from Insurance
with zipfile.ZipFile(DATA + 'Train.csv.zip', 'r') as z:
    z.extractall(DATA)
insurance = pd.read_csv(DATA + 'Train.csv')
print(f"Insurance → {insurance.shape}")

# ── ISP: remove 'id' column (paper explicitly states this) ─
isp = pd.read_csv(DATA + 'internet_service_churn.csv')
if 'id' in isp.columns:
    isp = isp.drop(columns=['id'])
    print(f"ISP       → {isp.shape}  (id column removed)")

# ── Telecom: load both files, combine for encoding ────────
# Paper: LabelEncoder applied to combined data
# then split using bigml-80=train, bigml-20=test
telecom_train = pd.read_csv(DATA + 'churn-bigml-80.csv')
telecom_test  = pd.read_csv(DATA + 'churn-bigml-20.csv')
telecom       = pd.concat([telecom_train, telecom_test],
                           ignore_index=True)
print(f"Telecom   → {telecom.shape}  (combined for encoding)")

churn_cols = {
    'Insurance' : 'labels',
    'ISP'       : 'churn',
    'Telecom'   : 'Churn'
}

datasets = {
    'Insurance' : insurance,
    'ISP'       : isp,
    'Telecom'   : telecom
}

print("\n✓ Datasets loaded")

Insurance → (33908, 17)
ISP       → (72274, 10)  (id column removed)
Telecom   → (3333, 20)  (combined for encoding)

✓ Datasets loaded


**Step1:Handle missing values**

In [3]:
# Paper: "Missing values were imputed using mean values
# for numerical columns and mode values for categorical columns"

cleaned = {}

for name, df in datasets.items():
    churn_col = churn_cols[name]
    df        = df.copy()
    before    = len(df)

    # Drop rows where label itself is missing
    df = df.dropna(subset=[churn_col])

    # Numeric → mean (paper says mean not median)
    num_cols = df.select_dtypes(include='number').columns
    num_cols = [c for c in num_cols if c != churn_col]
    df[num_cols] = df[num_cols].fillna(
        df[num_cols].mean()
    )

    # Categorical → mode
    obj_cols = df.select_dtypes(include='object').columns
    for col in obj_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    after = len(df)
    cleaned[name] = df
    print(f"{name:12} → before:{before:,}  "
          f"after:{after:,}  dropped:{before-after}")

print("\n✓ Missing values imputed (mean/mode)")

Insurance    → before:33,908  after:33,908  dropped:0
ISP          → before:72,274  after:72,274  dropped:0
Telecom      → before:3,333  after:3,333  dropped:0

✓ Missing values imputed (mean/mode)


**Step2:Standardize churn column to 0/1 integer**

In [4]:
for name, df in cleaned.items():
    churn_col = churn_cols[name]

    if df[churn_col].dtype == bool:
        df[churn_col] = df[churn_col].astype(int)
    elif df[churn_col].dtype == float:
        df[churn_col] = df[churn_col].astype(int)

    print(f"{name:12} → dtype:{df[churn_col].dtype}  "
          f"unique:{sorted(df[churn_col].unique())}")

print("\n✓ Churn standardized to 0/1")

Insurance    → dtype:int64  unique:[np.int64(0), np.int64(1)]
ISP          → dtype:int64  unique:[np.int64(0), np.int64(1)]
Telecom      → dtype:int64  unique:[np.int64(0), np.int64(1)]

✓ Churn standardized to 0/1


**Step3: Encode categorical columns:**

In [5]:
# Paper: "LabelEncoder was applied to combined data
# for Telecom — acceptable for encoding"
# For all datasets encoding done before split

from sklearn.preprocessing import LabelEncoder

encoded = {}

for name, df in cleaned.items():
    churn_col = churn_cols[name]
    df        = df.copy()
    obj_cols  = df.select_dtypes(
        include='object').columns.tolist()

    print(f"\n{name} — encoding {len(obj_cols)} columns:")
    for col in obj_cols:
        le      = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        print(f"  '{col}' encoded")

    encoded[name] = df

print("\n✓ Label encoding done on full data before split")


Insurance — encoding 0 columns:

ISP — encoding 0 columns:

Telecom — encoding 3 columns:
  'State' encoded
  'International plan' encoded
  'Voice mail plan' encoded

✓ Label encoding done on full data before split


**Step 4: Separate features and target**

In [6]:
X_dict = {}
y_dict = {}

for name, df in encoded.items():
    churn_col    = churn_cols[name]
    X_dict[name] = df.drop(columns=[churn_col])
    y_dict[name] = df[churn_col]

    print(f"{name:12} → X:{X_dict[name].shape}  "
          f"churn rate:{y_dict[name].mean()*100:.1f}%")

print("\n✓ Features and targets separated")

Insurance    → X:(33908, 16)  churn rate:11.7%
ISP          → X:(72274, 9)  churn rate:55.4%
Telecom      → X:(3333, 19)  churn rate:14.5%

✓ Features and targets separated


**Step 5:Apply RandomOverSampler**

In [7]:
# Paper's exact sequence per dataset:
#
# Insurance: ROS BEFORE split (paper admits leakage)
# "In the Insurance dataset, RandomOverSampler was applied
#  before train-test splitting, potentially introducing
#  minor leakage"
#
# ISP: normal split then ROS after
#
# Telecom: split using bigml-80/bigml-20 then ROS after

from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split

ros_splits = {}

# ── INSURANCE: ROS before split ───────────────────────────
print("Insurance — ROS BEFORE split (replicating paper exactly)")
X_ins = X_dict['Insurance']
y_ins = y_dict['Insurance']

print(f"  Before ROS → {X_ins.shape}  "
      f"churn:{y_ins.mean()*100:.1f}%")

ros = RandomOverSampler(random_state=42)
X_ins_ros, y_ins_ros = ros.fit_resample(X_ins, y_ins)
print(f"  After  ROS → {X_ins_ros.shape}  "
      f"churn:{y_ins_ros.mean()*100:.1f}%")
print(f"  Paper target: 59,882 rows")

# Now split after ROS
X_train, X_test, y_train, y_test = train_test_split(
    X_ins_ros, y_ins_ros,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y_ins_ros
)
ros_splits['Insurance'] = {
    'X_train' : X_train,
    'X_test'  : X_test,
    'y_train' : y_train,
    'y_test'  : y_test
}
print(f"  Train:{X_train.shape}  Test:{X_test.shape}")

# ── ISP: split then ROS after ─────────────────────────────
print("\nISP — split THEN ROS after")
X_isp = X_dict['ISP']
y_isp = y_dict['ISP']

X_train, X_test, y_train, y_test = train_test_split(
    X_isp, y_isp,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y_isp
)

ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

ros_splits['ISP'] = {
    'X_train' : X_train_ros,
    'X_test'  : X_test,
    'y_train' : y_train_ros,
    'y_test'  : y_test
}
print(f"  Before ROS → train:{X_train.shape}")
print(f"  After  ROS → train:{X_train_ros.shape}")
print(f"  Test:{X_test.shape}")
print(f"  Paper target: 50,375 rows")

# ── TELECOM: bigml-80=train, bigml-20=test, ROS after ─────
print("\nTelecom — bigml-80=train, bigml-20=test, ROS after")
n_train = len(telecom_train)

X_tel = X_dict['Telecom']
y_tel = y_dict['Telecom']

X_train = X_tel.iloc[:n_train].reset_index(drop=True)
X_test  = X_tel.iloc[n_train:].reset_index(drop=True)
y_train = y_tel.iloc[:n_train].reset_index(drop=True)
y_test  = y_tel.iloc[n_train:].reset_index(drop=True)

ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

ros_splits['Telecom'] = {
    'X_train' : X_train_ros,
    'X_test'  : X_test,
    'y_train' : y_train_ros,
    'y_test'  : y_test
}
print(f"  Before ROS → train:{X_train.shape}")
print(f"  After  ROS → train:{X_train_ros.shape}")
print(f"  Test:{X_test.shape}")
print(f"  Paper target: 5,700 rows")

print("\n✓ ROS applied per paper's exact sequence per dataset")

Insurance — ROS BEFORE split (replicating paper exactly)
  Before ROS → (33908, 16)  churn:11.7%
  After  ROS → (59882, 16)  churn:50.0%
  Paper target: 59,882 rows
  Train:(47905, 16)  Test:(11977, 16)

ISP — split THEN ROS after
  Before ROS → train:(57819, 9)
  After  ROS → train:(64080, 9)
  Test:(14455, 9)
  Paper target: 50,375 rows

Telecom — bigml-80=train, bigml-20=test, ROS after
  Before ROS → train:(2666, 19)
  After  ROS → train:(4556, 19)
  Test:(667, 19)
  Paper target: 5,700 rows

✓ ROS applied per paper's exact sequence per dataset


**Step 6: Scale numeric features:**

In [8]:
# Paper: "Numerical features were normalized to ensure
# uniform scaling, critical for models like CNN"
# Paper uses MinMaxScaler for Telecom (stated explicitly)

from sklearn.preprocessing import StandardScaler, MinMaxScaler

final_data = {}
scalers    = {}

for name in ['Insurance', 'ISP', 'Telecom']:
    X_train = pd.DataFrame(ros_splits[name]['X_train'])
    X_test  = pd.DataFrame(ros_splits[name]['X_test'])
    y_train = ros_splits[name]['y_train']
    y_test  = ros_splits[name]['y_test']

    # Paper uses MinMaxScaler for Telecom
    if name == 'Telecom':
        scaler = MinMaxScaler()
        print(f"Telecom    → MinMaxScaler (paper's method)")
    else:
        scaler = StandardScaler()
        print(f"{name:12} → StandardScaler")

    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    final_data[name] = {
        'X_train' : pd.DataFrame(X_train_sc,
                                  columns=X_train.columns),
        'X_test'  : pd.DataFrame(X_test_sc,
                                  columns=X_train.columns),
        'y_train' : y_train,
        'y_test'  : y_test
    }
    scalers[name] = scaler

    print(f"  train:{X_train_sc.shape}  "
          f"test:{X_test_sc.shape}")

print("\n✓ Scaling complete")

Insurance    → StandardScaler
  train:(47905, 16)  test:(11977, 16)
ISP          → StandardScaler
  train:(64080, 9)  test:(14455, 9)
Telecom    → MinMaxScaler (paper's method)
  train:(4556, 19)  test:(667, 19)

✓ Scaling complete


**Step#07-Verify sizes vs paper targets**

In [9]:
print("="*60)
print("  FINAL SIZES vs PAPER TARGETS")
print("="*60)

paper_targets = {
    'Insurance' : {'total_ros': 59882, 'note': 'ROS before split'},
    'ISP'       : {'total_ros': 50375, 'note': 'ROS after split'},
    'Telecom'   : {'total_ros': 5700,  'note': 'bigml-80 train'}
}

for name in ['Insurance', 'ISP', 'Telecom']:
    X_train = final_data[name]['X_train']
    X_test  = final_data[name]['X_test']
    y_train = final_data[name]['y_train']
    y_test  = final_data[name]['y_test']

    total   = len(X_train) + len(X_test)
    target  = paper_targets[name]['total_ros']
    diff    = abs(total - target)
    status  = "✅" if diff < 2000 else "⚠️"

    print(f"\n{name}")
    print(f"  Train size  : {len(X_train):,}")
    print(f"  Test size   : {len(X_test):,}")
    print(f"  Total       : {total:,}")
    print(f"  Paper target: {target:,}  {status}")
    print(f"  Difference  : {diff:,}")
    print(f"  Train churn : {pd.Series(y_train).mean()*100:.1f}%")
    print(f"  Test churn  : {pd.Series(y_test).mean()*100:.1f}%")
    print(f"  Features    : {X_train.shape[1]}")

print("\n✓ Verification complete")

  FINAL SIZES vs PAPER TARGETS

Insurance
  Train size  : 47,905
  Test size   : 11,977
  Total       : 59,882
  Paper target: 59,882  ✅
  Difference  : 0
  Train churn : 50.0%
  Test churn  : 50.0%
  Features    : 16

ISP
  Train size  : 64,080
  Test size   : 14,455
  Total       : 78,535
  Paper target: 50,375  ⚠️
  Difference  : 28,160
  Train churn : 50.0%
  Test churn  : 55.4%
  Features    : 9

Telecom
  Train size  : 4,556
  Test size   : 667
  Total       : 5,223
  Paper target: 5,700  ✅
  Difference  : 477
  Train churn : 50.0%
  Test churn  : 14.2%
  Features    : 19

✓ Verification complete


**Step 8: Save all processed data to Drive**

In [10]:
import pickle

# Save final data
final_path  = PROC + 'final_data.pkl'
scaler_path = PROC + 'scalers_final.pkl'

with open(final_path, 'wb') as f:
    pickle.dump(final_data, f)
print(f"✓ Final data saved → {final_path}")

with open(scaler_path, 'wb') as f:
    pickle.dump(scalers, f)
print(f"✓ Scalers saved    → {scaler_path}")

print("\n✓ Preprocessing complete — paper's exact sequence")
print("\nSequence used per dataset:")
print("  Insurance → impute → encode → ROS → split → scale")
print("  ISP       → impute → encode → split → ROS → scale")
print("  Telecom   → impute → encode → split(80/20) → ROS → scale")
print("\n✓ Next: update notebooks 03,04,05 to load final_data.pkl")

✓ Final data saved → /content/drive/MyDrive/churn_project/data/processed/final_data.pkl
✓ Scalers saved    → /content/drive/MyDrive/churn_project/data/processed/scalers_final.pkl

✓ Preprocessing complete — paper's exact sequence

Sequence used per dataset:
  Insurance → impute → encode → ROS → split → scale
  ISP       → impute → encode → split → ROS → scale
  Telecom   → impute → encode → split(80/20) → ROS → scale

✓ Next: update notebooks 03,04,05 to load final_data.pkl
